In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# 自动定位项目根目录：notebook 位于 notes/ 下，向上一级即为项目根
cwd = os.getcwd()
if os.path.basename(cwd) == 'notes':
    os.chdir('..')
print('当前工作目录:', os.getcwd())

当前工作目录: d:\github\prediction


In [ ]:
#mlp模型 （2）
import torch
from torch import nn
from d2l import torch as d2l

# ============================================================
# 1. 加载数据 (从预处理好的 CSV)
# ============================================================
df_train = pd.read_csv('data/df_train.csv', index_col='datetime', parse_dates=True)
df_test  = pd.read_csv('data/df_test.csv',  index_col='datetime', parse_dates=True)

# 确保 TEMP/HUMI 没有缺失值
df_train = df_train.dropna(subset=['pm_ave', 'TEMP', 'HUMI'])
df_test  = df_test.dropna(subset=['pm_ave', 'TEMP', 'HUMI'])

# ============================================================
# 2. 提取 4 个通道: pm_ave, TEMP, HUMI, hour
# ============================================================
# 训练集
pm_train   = torch.tensor(df_train['pm_ave'].values, dtype=torch.float32)
temp_train = torch.tensor(df_train['TEMP'].values, dtype=torch.float32)
humi_train = torch.tensor(df_train['HUMI'].values, dtype=torch.float32)
hour_train = torch.tensor(df_train.index.hour, dtype=torch.float32)

# 测试集
pm_test   = torch.tensor(df_test['pm_ave'].values, dtype=torch.float32)
temp_test = torch.tensor(df_test['TEMP'].values, dtype=torch.float32)
humi_test = torch.tensor(df_test['HUMI'].values, dtype=torch.float32)
hour_test = torch.tensor(df_test.index.hour, dtype=torch.float32)

print(f'训练序列长度: {len(pm_train)}  |  测试序列长度: {len(pm_test)}')

# ============================================================
# 3. 标准化 (每个通道用训练集统计量)
# ============================================================
means = torch.tensor([pm_train.mean(), temp_train.mean(), humi_train.mean(), hour_train.mean()])
stds  = torch.tensor([pm_train.std(),  temp_train.std(),  humi_train.std(),  hour_train.std()])

# 拼成多通道序列 (n, 4): [pm_ave, TEMP, HUMI, hour]
train_ch = torch.stack([pm_train, temp_train, humi_train, hour_train], dim=1)
test_ch  = torch.stack([pm_test, temp_test, humi_test, hour_test], dim=1)

train_ch = (train_ch - means) / stds
test_ch  = (test_ch  - means) / stds

# 分离出 pm_ave 标准化后的通道 (用于 label 和多步预测)
pm_train_s = train_ch[:, 0]
pm_test_s  = test_ch[:, 0]

# 保存 pm_ave 的标准化参数 (反标准化用)
pm_mean, pm_std = means[0], stds[0]

# ============================================================
# 4. 构建自回归特征 (过去 tau 步 × 4 通道 = 96 维输入)
# ============================================================
tau = 24       # 回看窗口
n_features = 4 # pm_ave, TEMP, HUMI, hour

n_train = len(train_ch)
features_train = torch.zeros((n_train - tau, tau * n_features))
for t in range(tau):
    for ch in range(n_features):
        features_train[:, t * n_features + ch] = train_ch[t: n_train - tau + t, ch]
labels_train = pm_train_s[tau:].reshape((-1, 1))

# 数据迭代器
batch_size = 16
train_iter = d2l.load_array((features_train, labels_train), batch_size, is_train=True)

# ============================================================
# 5. 定义网络 + 损失
#    网络结构: 96 → 256 → 20 → 1
# ============================================================
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def get_net():
    net = nn.Sequential(
        nn.Linear(tau * n_features, 256),  # 96 → 256
        nn.ReLU(),
        nn.Linear(256, 20),               # 256 → 20
        nn.ReLU(),
        nn.Linear(20, 1)                   # 20 → 1
    )
    net.apply(init_weights)
    return net

loss = nn.MSELoss(reduction='none')

# ============================================================
# 6. 训练 (只用 train 数据, 带 L2 正则化)
# ============================================================
def train(net, train_iter, loss, epochs, lr, weight_decay=1e-4):
    # 用 Adam 自带的 weight_decay 实现 L2 正则化 (稳定, 不会压制 MSE)
    trainer = torch.optim.Adam(net.parameters(), lr, weight_decay=weight_decay)
    for epoch in range(epochs):
        net.train()
        for X, y in train_iter:
            trainer.zero_grad()
            l = loss(net(X), y)
            l.mean().backward()  # 用 mean 而非 sum, 使 loss 不随 batch_size 变化
            trainer.step()
        print(f'epoch {epoch + 1}, '
              f'loss: {d2l.evaluate_loss(net, train_iter, loss):f}')

net = get_net()
train(net, train_iter, loss, 20, 0.001)

# ============================================================
# 7. 测试: 前 24 条公开, 剩余多步预测
#    TEMP/HUMI/hour 是外生变量 (测试集已知), 只有 pm_ave 需要预测
# ============================================================
n_public = 24

test_preds = torch.zeros(len(test_ch))
test_preds[:n_public] = pm_test_s[:n_public]

with torch.no_grad():
    for i in range(n_public, len(test_ch)):
        # 取过去 tau 步全部 4 个通道 (外生变量用真实值)
        window = test_ch[i - tau:i].clone()  # (tau, 4)
        # pm_ave 通道: 公开区间用真实值, 之后用预测值替代
        for t in range(tau):
            idx = i - tau + t
            if idx >= n_public:
                window[t, 0] = test_preds[idx]
        test_preds[i] = net(window.reshape(1, -1))

# 反标准化回原始量纲 (μg/m³)
preds_all  = (test_preds * pm_std + pm_mean).detach()
actual_all = (pm_test_s  * pm_std + pm_mean).detach()

print(f'\n公开起点: 前 {n_public} 条  |  预测区间: 第 {n_public+1} ~ {len(test_ch)} 条 (共 {len(test_ch)-n_public} 条)')

In [ ]:
# --- 单步预测图 ---
# 前 24 条: predicted = actual (公开数据)
# 第 25 条起: 多步预测 (pm_ave 用预测值, TEMP/HUMI/hour 用真实值)
test_time = torch.arange(1, len(test_ch) + 1, dtype=torch.float32)
d2l.plot([test_time, test_time],
         [actual_all.numpy(), preds_all.numpy()],
         'test timestep', 'PM2.5 (μg/m³)',
         legend=['actual', '1-step preds'], figsize=(10, 4))

In [ ]:
# --- 预测结果表格 + MSE ---
# 只展示预测区间 (第 25 条起, 共 377 条)
df_result = pd.DataFrame({
    'timestamp': df_test.index[n_public:],
    'predicted': preds_all[n_public:].numpy(),
    'actual':    actual_all[n_public:].numpy(),
})
df_result['error'] = df_result['actual'] - df_result['predicted']

# 逐条计算均方误差
err = df_result['error']
mse = (err ** 2).mean()

print(f'预测样本数: {len(df_result)}')
print(f'MSE = {mse:.2f}')
df_result